# Operator Overloading and Friend Functions

## Review: Operator Overloading; Compound Assignment Operators; The Left-Hand Side Problem; Friend Functions; Overloading `<<`; Overloading `>>`; Putting It All Together

<br><br><br><br><br>
<br><br><br><br><br>

# 1. Review: Operator Overloading

### Last time, we saw that C++ lets us define how operators like `+`, `*`, and `==` work for our own classes. Let's recall the key ideas using a `Fraction` class.

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////reviewFraction.cpp
// Example 1a - Review: Fraction class with +, ==
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    int getNum() const { return this->num_; }
    int getDen() const { return this->den_; }
    
    Fraction operator+(const Fraction& other) const {
        int newNum = this->num_ * other.den_ + other.num_ * this->den_;
        int newDen = this->den_ * other.den_;
        return Fraction(newNum, newDen);
    }
    
    bool operator==(const Fraction& other) const {
        return this->num_ * other.den_ == other.num_ * this->den_;
    }
    
    void display() const {
        std::cout << this->num_ << "/" << this->den_;
    }
};

int main(){
    Fraction a(1, 2);
    Fraction b(1, 3);
    Fraction c = a + b;
    
    c.display();                          // what prints?
    std::cout << "\n";
    
    std::cout << (c == Fraction(5, 6)) << "\n";  // what prints?
}

<br><br><br><br><br>
<br><br><br><br><br>

### Recall:
- ### `a + b` is really `a.operator+(b)`. The left operand is `this`, the right operand is `other`.
- ### `operator+` returns a **new** `Fraction`. It does not modify `a` or `b`.
- ### `operator==` returns `bool`. We compare cross-multiplied products to avoid reducing fractions.

<br><br><br><br><br>
<br><br><br><br><br>

### Notice the annoyance: to print a `Fraction`, we have to call `c.display()`. We can't just write `std::cout << c`. Wouldn't it be nice if we could?

### We'll get there, but first let's cover some other operators and an important concept called **friend functions**.

<br><br><br><br><br>
<br><br><br><br><br>

# 2. Compound Assignment Operators (`+=`, `-=`)

### As we learn more operators today, you'll notice each one has a slightly different signature. Different return types, `const` vs not, member vs non-member. **Don't try to memorize these.** Instead, ask yourself: *what does this operator do with built-in types?* The answer tells you how to write it.

### Let's start with `+=`. Think about how it works with `int`s:

In [ ]:
////compoundReview.cpp
// Example 2a - Compound assignment with built-in types
#include<iostream>

int main(){
    int x = 10;
    int y = 3;
    int z = 1;
    
    x += y;   // Does x change? Does y change? Is there a new integer created?
    std::cout << "x = " << x << "\n";
    std::cout << "y = " << y << "\n";
    
    // chaining works too:
    x = 10;
    x += y += z; // What is the order here? Hint: assignment starts on the...(left or right?)
    std::cout << "y = " << y << "\n";
    std::cout << "x = " << x << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Now we want to write `operator+=` for `Fraction`. Based on what you just saw with `int`s:

- ### Does `+=` create a new value, or change an existing one? So should the function be `const`?
- ### In the chaining example, `x += y += z`, which `+=` happens first?
- ### What must `y += z` give back in order for `x += ___` to work?

In [ ]:
////fractionPlusEquals.cpp
// Example 2b - Overloading +=
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    void display() const {
        std::cout << this->num_ << "/" << this->den_;
    }
    
    // += modifies this object and returns a reference to it
   /* return type */ operator+=(/* parameter */) {
       // Let's code together!
    }
};

int main(){
    Fraction a(1, 2);
    Fraction b(1, 3);
    
    a += b;   // What is a now?
    a.display();
    std::cout << "\n";
    
    b.display();  // What is b?
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### So to recap: every part of the signature follows from how `+=` works with built-in types.
- ### It changes the left-hand side, so it **cannot** be `const`.
- ### `y += z` must return the modified `y` so that `x += y += z` works. That means returning `*this` by **reference**.
- ### It does not create a new object, so we return `Fraction&`, not `Fraction`.

<br><br><br><br><br>
<br><br><br><br><br>

### You try! Here's a starter `Compl` class for complex numbers. Recall: $(a+bi) + (c+di) = (a+c) + (b+d)i$.

### Add `operator+` and `operator+=` to the class.

In [ ]:
////complPlusEquals.cpp
// Example 2c - You try! Add operator+ and operator+= to this class.
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    double getRe() const { return this->re_; }
    double getIm() const { return this->im_; }
    
    void display() const {
        if(this->im_ >= 0)
            std::cout << this->re_ << " + " << this->im_ << "i";
        else
            std::cout << this->re_ << " - " << -this->im_ << "i";
    }
    
    // write operator+ here
    
    
    // write operator+= here
    
    
};

int main(){
    Compl z1(3, 4);    // 3 + 4i
    Compl z2(1, -2);   // 1 - 2i
    
    Compl z3 = z1 + z2;
    z3.display();       // should print: 4 + 2i
    std::cout << "\n";
    
    z1 += z2;
    z1.display();       // should print: 4 + 2i
    std::cout << "\n";
    
    z2.display();       // should print: 1 - 2i (unchanged)
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### A nice trick: once you have `operator+=`, you can write `operator+` in terms of it. This avoids duplicating the addition logic in two places. The convention is to write `+=` first (the real logic), then write `+` as a copy-then-modify.

In [ ]:
////plusFromPlusEquals.cpp
// Example 2d - Writing operator+ in terms of operator+=
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    void display() const {
        if(this->im_ >= 0)
            std::cout << this->re_ << " + " << this->im_ << "i";
        else
            std::cout << this->re_ << " - " << -this->im_ << "i";
    }
    
    // write operator+= here
    
    // write operator+ here
    

};

int main(){
    Compl z1(3, 4);
    Compl z2(1, -2);
    
    Compl z3 = z1 + z2;
    z3.display();
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

# 3. The Left-Hand Side Problem

### When we write `operator+` as a **member function**, the left operand is always `*this`. That means `*this` must be an object of the class.

### This works fine when both operands are the same type. But what if we want to mix types?

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////leftHandProblem.cpp
// Example 3a - The left-hand side problem
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    void display() const {
        if(this->im_ >= 0)
            std::cout << this->re_ << " + " << this->im_ << "i";
        else
            std::cout << this->re_ << " - " << -this->im_ << "i";
    }
    
    // Member function: Compl + double works
    Compl operator+(double d) const {
        return Compl(this->re_ + d, this->im_);
    }
};

int main(){
    Compl z(3, 4);
    
    Compl r1 = z + 2.0;   // OK
    r1.display();
    std::cout << "\n";
    
    // Compl r2 = 2.0 + z;   // will this work? why or why not?
}

<br><br><br><br><br>
<br><br><br><br><br>

### `z + 2.0` works. It's really `z.operator+(2.0)`, and `z` is a `Compl`, so the member function runs.

### But what about `2.0 + z`? Think about how member operator overloading has worked so far:
- ### Which object does the member function get called **on**?
- ### So for `2.0 + z`, where would the member function need to live?
- ### Is that something we can do?

<br><br><br><br><br>
<br><br><br><br><br>

# 4. Friend Functions

### So we need a function that is **not** a member of our class (because the left operand isn't our object), but it still needs to access private data members. How do we do that?

### Think about it this way: who do you share private things with? Not strangers, not enemies... you share them with your **friends**.

### C++ uses exactly this metaphor. Bjarne Stroustrup (the creator of C++) introduced the `friend` keyword to let a class say: "This specific function is allowed to access my private members, even though it's not one of my member functions."

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////friendFunction.cpp
// Example 4a - Solving the left-hand side problem with a friend function
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    void display() const {
        if(this->im_ >= 0)
            std::cout << this->re_ << " + " << this->im_ << "i";
        else
            std::cout << this->re_ << " - " << -this->im_ << "i";
    }
    
    // Member function: Compl + double
    Compl operator+(double d) const {
        return Compl(this->re_ + d, this->im_);
    }
    
    // Friend function: double + Compl
    // Declared inside the class, but NOT a member function
    friend Compl operator+(double d, const Compl& z);
};

// Defined outside the class, no Compl:: prefix!
Compl operator+(double d, const Compl& z){
    return Compl(d + z.re_, z.im_);  // can access re_ and im_ because it's a friend
}

int main(){
    Compl z(3, 4);
    
    Compl r1 = z + 2.0;   // calls member function
    r1.display();
    std::cout << "\n";
    
    Compl r2 = 2.0 + z;   // calls friend function
    r2.display();
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Look at the friend function we just wrote. What do you notice?
- ### Where does the `friend` declaration go? Where does the function definition go?
- ### Does the friend function have a `this` pointer? How does it get access to both operands?
- ### When we defined the function outside the class, did we write `Compl::operator+`? Why or why not?

### One more thing: a friend function accesses private members directly. But what if the class had **getters** for the data we need? Then we wouldn't need `friend` at all. We'll see an example of this soon.

<br><br><br><br><br>
<br><br><br><br><br>

### Let's try this with the `Fraction` class. We want `3 * Fraction(1, 2)` to give us `Fraction(3, 2)`.

### That's `int * Fraction`, so the left operand is an `int`. Let's write it together.

In [ ]:
////fractionFriend.cpp
// Example 4b - Let's write int * Fraction together
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    void display() const {
        std::cout << this->num_ << "/" << this->den_;
    }
    
    // write friend operator* here so int * Fraction works
    
    
};

// define the friend function here


int main(){
    Fraction a(1, 2);
    
    Fraction r1 = 3 * a;    // should give 3/2
    r1.display();
    std::cout << "\n";
    
    Fraction r2 = 5 * Fraction(2, 3);   // should give 10/3
    r2.display();
    std::cout << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### You try! Write a friend function `operator==` so that we can compare a `double` with a `Compl`.

### A `double` should be equal to a `Compl` if the real part matches and the imaginary part is 0.

In [ ]:
////complFriendEqual.cpp
// Example 4c - You try! Write a friend operator== so that double == Compl works.
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    // Member ==: Compl == Compl
    bool operator==(const Compl& other) const {
        return this->re_ == other.re_ && this->im_ == other.im_;
    }
    
    // Declare friend operator== here so double == Compl works
    friend bool operator==(double n, const Compl& other);
    
};

// Define the friend function here
friend bool operator==(double n, const Compl& other){
    return n == other.re_ && other.im_ == 0; 
    
}

int main(){
    Compl z1(5, 0);
    Compl z2(3, 4);
    
    std::cout << (z1 == z2) << "\n";     // 0 (false)
    std::cout << (5.0 == z1) << "\n";    // should be 1 (true)
    std::cout << (5.0 == z2) << "\n";    // should be 0 (false)
    std::cout << (3.0 == z2) << "\n";    // should be 0 (false), imaginary part isn't 0
}

wrote file complFriendEqual.cpp


complFriendEqual.cpp:23:1: error: ‘friend’ used outside of class
   23 | friend bool operator==(double n, const Compl& other){
      | ^~~~~~
      | ------


<br><br><br><br><br>
<br><br><br><br><br>

# 5. Overloading `<<` (The Stream Insertion Operator)

### We've been using `display()` methods to print our objects. But it would be much nicer to write:
### `std::cout << z << "\n";`

### When we write `std::cout << 5`, there are two operands. What is the left operand? What is the right operand?

### Now think about `std::cout << z` where `z` is a `Compl`. If we wanted to overload `operator<<`:
- ### What type is the left operand?
- ### Is the left operand a `Compl`? So can this be a member function of `Compl`?
- ### We just ran into this exact situation. What did we use to solve it?

<br><br><br><br><br>
<br><br><br><br><br>

In [ ]:
////fractionStream.cpp
// Example 5a - Overloading << with a friend function (Fraction)
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    Fraction operator+(const Fraction& other) const {
        int newNum = this->num_ * other.den_ + other.num_ * this->den_;
        int newDen = this->den_ * other.den_;
        return Fraction(newNum, newDen);
    }
    
    bool operator==(const Fraction& other) const {
        return this->num_ * other.den_ == other.num_ * this->den_;
    }
    
    // Declare the friend function inside the class
    friend std::ostream& operator<<(std::ostream& out, const Fraction& f);
};

// Define outside the class
std::ostream& operator<<(std::ostream& out, const Fraction& f){
    out << f.num_ << "/" << f.den_;
    return out;
}

int main(){
    Fraction a(1, 2);
    Fraction b(1, 3);
    Fraction c = a + b;
    
    std::cout << a << " + " << b << " = " << c << "\n";
    std::cout << (c == Fraction(5, 6)) << "\n";
    operator<<(std::cout,a);
}


<br><br><br><br><br>
<br><br><br><br><br>

### Let's look at the signature: `std::ostream& operator<<(std::ostream& out, const Fraction& f)`

### We can reason through every part of this, just like we did with `+=`:
- ### `std::cout << a << b` chains left to right. What must `std::cout << a` return so that `<< b` can follow? (Think back to how we figured out the return type for `+=`.)
- ### Does `<<` change the `Fraction` being printed? So should the `Fraction` parameter be `const`?
- ### `ostream` objects (like `std::cout`) cannot be copied. So should `out` be passed by value or by reference?
- ### Why did we use `friend` here? 

### But notice: the `Fraction` class has `getNum()` and `getDen()`. We could have written this as a plain non-member function (no `friend` needed) that just calls those getters. Both approaches work. Using getters keeps the class's private data more protected. Using `friend` is more direct. You'll see both in practice.

<br><br><br><br><br>
<br><br><br><br><br>

### You try! Write `operator<<` for the `Compl` class.

### It should print in the form `a + bi` or `a - bi` (if the imaginary part is negative).

In [1]:
////complStream.cpp
// Example 5b - You try! Write operator<< for Compl.
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    Compl operator+(const Compl& other) const {
        return Compl(this->re_ + other.re_, this->im_ + other.im_);
    }
    
    // Declare and define operator<< as a friend
    friend std::ostream& operator<<(std::ostream& os, const Compl& z);
    
};

// Define operator<< here
std::ostream& operator<<(std::ostream& os, const Compl& z){

    os << z.re_;
    if(z.im_ >= 0){
        os << "+" << z.im_ << "i";
    }
    else{
        os << " - " << (-z.im_) << "i";
    }
    return os;
}

int main(){
    Compl z1(3, 4);
    Compl z2(1, -2);
    
    std::cout << z1 << "\n";              // should print: 3 + 4i
    std::cout << z2 << "\n";              // should print: 1 - 2i
    std::cout << z1 + z2 << "\n";         // should print: 4 + 2i
    std::cout << z1 << " + " << z2 << " = " << z1 + z2 << "\n";
}

wrote file complStream.cpp
$> g++   complStream.cpp    -o complStream
$> ./complStream 
3+4i
1 - 2i
4+2i
3+4i + 1 - 2i = 4+2i


<br><br><br><br><br>
<br><br><br><br><br>

# 6. Overloading `>>` (The Stream Extraction Operator)

### Just like `<<` puts data **into** a stream, `>>` reads data **from** a stream. Let's look at how it works with built-in types first:

In [ ]:
////extractionReview.cpp
// Example 6a - Stream extraction with built-in types
#include<iostream>

int main(){
    int x = 0;
    std::cout << "Enter a number: ";
    std::cin >> x;
    std::cout << "x is now " << x << "\n";
    
    int y = 0;
    int z = 0;
    std::cout << "Enter two numbers: ";
    std::cin >> y >> z;
    std::cout << "y = " << y << ", z = " << z << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### You've now written `operator<<`. How is `>>` different?
- ### What happened to `x` after `std::cin >> x`? What about `y` and `z`?
- ### Compare that to `<<`, which just prints a value. Which one changes the object?
- ### Based on that, how would you expect the signature of `operator>>` to differ from `operator<<`?

In [ ]:
////fractionInput.cpp
// Example 6b - Overloading >> for Fraction
#include<iostream>

class Fraction{
private:
    int num_;
    int den_;
public:
    Fraction(int n, int d) : num_(n), den_(d) {}
    
    friend std::ostream& operator<<(std::ostream& out, const Fraction& f){
        out << f.num_ << "/" << f.den_;
        return out;
    }
    
    // >> reads two integers into the fraction
    friend std::istream& operator>>(std::istream& in, Fraction& f){
        in >> f.num_ >> f.den_;
        return in;
    }
};

int main(){
    Fraction f(0,1);
    std::cout << "Enter numerator and denominator: ";
    std::cin >> f;
    std::cout << "You entered: " << f << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Same chaining logic as before: `std::cin >> a >> b` works because `operator>>` returns the stream by reference.

### One new thing to notice: in this example, the friend functions are **defined inside** the class body. This is allowed as a shortcut. The function is still not a member, even though it appears inside the class.

<br><br><br><br><br>
<br><br><br><br><br>

### You try! Write `operator<<` and `operator>>` for the `Compl` class. You've already done `<<` for Compl before, so this time add `>>` as well. It should read two doubles (real part, then imaginary part).

In [ ]:
////complInput.cpp
// Example 6c - You try! Write operator<< and operator>> for Compl.
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    // Declare friend operator<< here (print as a + bi or a - bi)
    
    
    // Declare friend operator>> here (read real part, then imaginary part)
    
    
};

// Define operator<< here


// Define operator>> here


int main(){
    Compl z(0,0);
    std::cout << "Enter real and imaginary parts: ";
    std::cin >> z;
    std::cout << "You entered: " << z << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

# 7. Putting It All Together

### You've built pieces of the `Compl` class throughout today. Now write the whole class from scratch with all of these operators:

- ### `operator+` (Compl + Compl)
- ### `operator+=`
- ### `operator==`
- ### `friend operator+` (double + Compl)
- ### `friend operator<<`
- ### `friend operator>>`

### The `main` function below is your spec. If your code compiles and produces the expected output, you've got it.

In [ ]:
////complFull.cpp
// Example 7a - Write the full Compl class!
#include<iostream>

class Compl{
private:
    double re_;
    double im_;
public:
    Compl(double r, double i) : re_(r), im_(i) {}
    
    // Write all your operators here:
    // operator+ (Compl + Compl)
    // operator+=
    // operator==
    // friend operator+ (double + Compl)
    // friend operator<<
    // friend operator>>
    
    // Also try:
    // operator-
    // operator-=
    // operator* (first try to see how the multiplication works out by hand)
    // operator*=
    
};

// Define any non-member functions here


int main(){
    Compl a(1, 2);
    Compl b(3, -1);
    
    std::cout << a << "\n";                    // 1 + 2i
    std::cout << b << "\n";                    // 3 - 1i
    std::cout << a + b << "\n";                // 4 + 1i
    
    a += b;
    std::cout << a << "\n";                    // 4 + 1i
    std::cout << (a == Compl(4, 1)) << "\n";   // 1
    std::cout << (a == b) << "\n";             // 0
    
    Compl c = 10.0 + b;
    std::cout << c << "\n";                    // 13 - 1i
    
    Compl d(0,0);
    std::cout << "Enter real and imaginary parts: ";
    std::cin >> d;
    std::cout << "You entered: " << d << "\n";
}

<br><br><br><br><br>
<br><br><br><br><br>

### Try it! Complex multiplication

### How do you multiply two complex numbers? Work it out by hand first:
### $(a + bi) \times (c + di) = \;?$

### Hint: use FOIL, and remember that $i^2 = -1$.

### Once you have the formula, add `operator*` to your `Compl` class and test it:
### $(1 + 2i) \times (3 - 1i) = \;?$